# 🔬 Exploratory Data Analysis - QM9 Molecular Dataset

This notebook performs exploratory data analysis on the QM9 molecular dataset to understand the data distribution, molecular properties, and characteristics that will inform our VAE model design.

## 📋 Table of Contents
1. [Dataset Loading and Overview](#dataset-loading)
2. [Molecular Size Distribution](#molecular-size)
3. [Atomic Composition Analysis](#atomic-composition)
4. [Coordinate Distribution Analysis](#coordinate-distribution)
5. [Molecular Geometry Analysis](#geometry-analysis)
6. [Visualization of Sample Molecules](#visualization)
7. [Statistical Summary](#statistics)


## 📚 Imports and Setup

In [ ]:
# Standard libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Chemistry libraries
from rdkit import Chem
from rdkit.Chem import rdmolfiles, Descriptors, Draw
from rdkit.Chem.rdMolDescriptors import CalcMolFormula

# 3D visualization
import py3Dmol
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Add src to path for custom modules
sys.path.append('../src')
from data_loader import QM9Dataset
from preprocess import MolecularPreprocessor
from visualize import MolecularVisualizer

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All imports successful!")

## 📂 Dataset Loading and Overview {#dataset-loading}

In [ ]:
# Set up paths
data_path = '../data'

# Check if dataset exists
if not os.path.exists(data_path):
    print("❌ Dataset not found. Please download QM9 dataset to ../data/")
    print("📥 Download from: https://figshare.com/collections/Quantum_chemistry_structures_and_properties_of_134k_molecules/978904")
else:
    print("✅ Dataset directory found!")
    
    # Create sample dataset for analysis
    try:
        dataset = QM9Dataset(data_path, max_atoms=29)
        print(f"📊 Loaded {len(dataset)} molecules from dataset")
        
        # Get sample batch for analysis
        sample_size = min(1000, len(dataset))
        sample_indices = np.random.choice(len(dataset), sample_size, replace=False)
        
        molecules_data = []
        for idx in sample_indices:
            mol_data = dataset[idx]
            molecules_data.append({
                'atomic_nums': mol_data['atomic_nums'].numpy(),
                'coordinates': mol_data['coordinates'].numpy(),
                'mask': mol_data['mask'].numpy(),
                'num_atoms': mol_data['num_atoms'].item()
            })
        
        print(f"📈 Analyzing {len(molecules_data)} molecules")
        
    except Exception as e:
        print(f"❌ Error loading dataset: {e}")
        print("🔄 Creating synthetic dataset for demonstration...")
        
        # Create synthetic dataset for demonstration
        molecules_data = []
        atom_types = [1, 6, 7, 8, 9]  # H, C, N, O, F
        
        for i in range(1000):
            num_atoms = np.random.randint(3, 20)
            atomic_nums = np.random.choice(atom_types, num_atoms)
            coordinates = np.random.randn(num_atoms, 3) * 2.0
            
            molecules_data.append({
                'atomic_nums': atomic_nums,
                'coordinates': coordinates,
                'mask': np.ones(num_atoms),
                'num_atoms': num_atoms
            })
        
        print(f"📈 Created synthetic dataset with {len(molecules_data)} molecules")

## 📏 Molecular Size Distribution {#molecular-size}

In [ ]:
# Extract molecular sizes
mol_sizes = [mol['num_atoms'] for mol in molecules_data]

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Molecular Size Distribution', fontsize=16, fontweight='bold')

# Histogram
axes[0, 0].hist(mol_sizes, bins=30, alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Number of Atoms')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Molecular Sizes')
axes[0, 0].grid(True, alpha=0.3)

# Box plot
axes[0, 1].boxplot(mol_sizes, vert=True)
axes[0, 1].set_ylabel('Number of Atoms')
axes[0, 1].set_title('Box Plot of Molecular Sizes')
axes[0, 1].grid(True, alpha=0.3)

# Cumulative distribution
sorted_sizes = np.sort(mol_sizes)
cumulative = np.arange(1, len(sorted_sizes) + 1) / len(sorted_sizes)
axes[1, 0].plot(sorted_sizes, cumulative, linewidth=2)
axes[1, 0].set_xlabel('Number of Atoms')
axes[1, 0].set_ylabel('Cumulative Probability')
axes[1, 0].set_title('Cumulative Distribution')
axes[1, 0].grid(True, alpha=0.3)

# Statistics table
stats_text = f"""
Statistics:
Mean: {np.mean(mol_sizes):.2f}
Median: {np.median(mol_sizes):.2f}
Std: {np.std(mol_sizes):.2f}
Min: {np.min(mol_sizes)}
Max: {np.max(mol_sizes)}
25th percentile: {np.percentile(mol_sizes, 25):.2f}
75th percentile: {np.percentile(mol_sizes, 75):.2f}
"""
axes[1, 1].text(0.1, 0.5, stats_text, fontsize=12, verticalalignment='center')
axes[1, 1].set_xlim(0, 1)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].axis('off')
axes[1, 1].set_title('Size Statistics')

plt.tight_layout()
plt.show()

print(f"📊 Molecular Size Statistics:")
print(f"   Mean atoms: {np.mean(mol_sizes):.2f}")
print(f"   Median atoms: {np.median(mol_sizes):.2f}")
print(f"   Std deviation: {np.std(mol_sizes):.2f}")
print(f"   Range: {np.min(mol_sizes)} - {np.max(mol_sizes)} atoms")

## ⚛️ Atomic Composition Analysis {#atomic-composition}

In [ ]:
# Count atom types across all molecules
atom_counts = {1: 0, 6: 0, 7: 0, 8: 0, 9: 0}  # H, C, N, O, F
atom_names = {1: 'Hydrogen', 6: 'Carbon', 7: 'Nitrogen', 8: 'Oxygen', 9: 'Fluorine'}

for mol in molecules_data:
    for atomic_num in mol['atomic_nums']:
        if atomic_num in atom_counts:
            atom_counts[atomic_num] += 1

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Atomic Composition Analysis', fontsize=16, fontweight='bold')

# Pie chart of atom types
labels = [atom_names[num] for num in atom_counts.keys()]
sizes = list(atom_counts.values())
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']

axes[0, 0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0, 0].set_title('Atom Type Distribution')

# Bar chart of atom counts
atom_nums = list(atom_counts.keys())
counts = list(atom_counts.values())
element_symbols = [ Chem.GetPeriodicTable().GetElementSymbol(num) for num in atom_nums ]

bars = axes[0, 1].bar(element_symbols, counts, color=colors)
axes[0, 1].set_xlabel('Element')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Total Atom Counts')
axes[0, 1].grid(True, alpha=0.3)

# Add value labels on bars
for bar, count in zip(bars, counts):
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                    f'{count:,}', ha='center', va='bottom')

# Atoms per molecule distribution
atoms_per_mol = {}
for mol in molecules_data:
    unique_atoms = set(mol['atomic_nums'])
    num_unique = len(unique_atoms)
    atoms_per_mol[num_unique] = atoms_per_mol.get(num_unique, 0) + 1

unique_counts = list(atoms_per_mol.keys())
unique_freqs = list(atoms_per_mol.values())

axes[1, 0].bar(unique_counts, unique_freqs, alpha=0.7, color='skyblue', edgecolor='black')
axes[1, 0].set_xlabel('Number of Different Atom Types')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Atom Type Diversity per Molecule')
axes[1, 0].grid(True, alpha=0.3)

# Composition statistics table
total_atoms = sum(atom_counts.values())
composition_text = "Element Composition:\n\n"
for num, count in atom_counts.items():
    percentage = (count / total_atoms) * 100
    composition_text += f"{atom_names[num]}: {count:,} ({percentage:.1f}%)\n"

axes[1, 1].text(0.1, 0.5, composition_text, fontsize=11, verticalalignment='center')
axes[1, 1].set_xlim(0, 1)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].axis('off')
axes[1, 1].set_title('Composition Summary')

plt.tight_layout()
plt.show()

print(f"🧬 Atomic Composition Summary:")
print(f"   Total atoms analyzed: {total_atoms:,}")
for num, count in atom_counts.items():
    percentage = (count / total_atoms) * 100
    print(f"   {atom_names[num]:<10}: {count:>8,} ({percentage:>5.1f}%)")

## 📐 Coordinate Distribution Analysis {#coordinate-distribution}

In [ ]:
# Collect all coordinates
all_coords = []
coord_ranges = []

for mol in molecules_data:
    coords = mol['coordinates']
    # Only consider non-zero coordinates (valid atoms)
    valid_coords = coords[mol['mask'] > 0.5]
    if len(valid_coords) > 0:
        all_coords.extend(valid_coords)
        
        # Calculate coordinate range for this molecule
        coord_range = np.max(valid_coords, axis=0) - np.min(valid_coords, axis=0)
        coord_ranges.append(coord_range)

all_coords = np.array(all_coords)
coord_ranges = np.array(coord_ranges)

# Create visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('3D Coordinate Distribution Analysis', fontsize=16, fontweight='bold')

# X, Y, Z coordinate distributions
coord_labels = ['X Coordinate', 'Y Coordinate', 'Z Coordinate']
coord_data = [all_coords[:, 0], all_coords[:, 1], all_coords[:, 2]]
colors = ['red', 'green', 'blue']

for i, (label, data, color) in enumerate(zip(coord_labels, coord_data, colors)):
    axes[0, i].hist(data, bins=50, alpha=0.7, color=color, edgecolor='black')
    axes[0, i].set_xlabel(f'{label} (Å)')
    axes[0, i].set_ylabel('Frequency')
    axes[0, i].set_title(f'{label} Distribution')
    axes[0, i].grid(True, alpha=0.3)
    
    # Add statistics
    mean_val = np.mean(data)
    std_val = np.std(data)
    axes[0, i].axvline(mean_val, color='black', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    axes[0, i].axvline(mean_val + std_val, color='orange', linestyle=':', label=f'±1σ: {std_val:.2f}')
    axes[0, i].axvline(mean_val - std_val, color='orange', linestyle=':')
    axes[0, i].legend()

# Molecular size ranges
range_labels = ['X Range', 'Y Range', 'Z Range']
range_data = [coord_ranges[:, 0], coord_ranges[:, 1], coord_ranges[:, 2]]

for i, (label, data, color) in enumerate(zip(range_labels, range_data, colors)):
    axes[1, i].hist(data, bins=30, alpha=0.7, color=color, edgecolor='black')
    axes[1, i].set_xlabel(f'{label} (Å)')
    axes[1, i].set_ylabel('Frequency')
    axes[1, i].set_title(f'Molecular {label} Distribution')
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print coordinate statistics
print(f"📐 Coordinate Statistics:")
for i, label in enumerate(['X', 'Y', 'Z']):
    data = coord_data[i]
    print(f"   {label} - Mean: {np.mean(data):.3f}, Std: {np.std(data):.3f}, Range: [{np.min(data):.3f}, {np.max(data):.3f}]")

print(f"\n📏 Molecular Size Statistics:")
for i, label in enumerate(['X', 'Y', 'Z']):
    data = range_data[i]
    print(f"   {label} Range - Mean: {np.mean(data):.3f}, Std: {np.std(data):.3f}, Max: {np.max(data):.3f}")

## 🔍 Molecular Geometry Analysis {#geometry-analysis}

In [ ]:
# Analyze molecular geometry
bond_lengths = []
angles = []
center_of_mass_distances = []

def calculate_distance(coord1, coord2):
    return np.linalg.norm(coord1 - coord2)

def calculate_angle(coord1, coord2, coord3):
    v1 = coord1 - coord2
    v2 = coord3 - coord2
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return np.arccos(np.clip(cos_angle, -1, 1)) * 180 / np.pi

# Covalent radii for bond detection
covalent_radii = {1: 0.31, 6: 0.76, 7: 0.71, 8: 0.66, 9: 0.57}

for mol in molecules_data:
    coords = mol['coordinates']
    atomic_nums = mol['atomic_nums']
    mask = mol['mask']
    
    # Get valid atoms
    valid_indices = np.where(mask > 0.5)[0]
    
    if len(valid_indices) < 2:
        continue
    
    valid_coords = coords[valid_indices]
    valid_atomic_nums = atomic_nums[valid_indices]
    
    # Calculate center of mass
    center = np.mean(valid_coords, axis=0)
    
    # Distance from center of mass
    for coord in valid_coords:
        center_of_mass_distances.append(calculate_distance(coord, center))
    
    # Detect bonds and calculate bond lengths
    for i in range(len(valid_coords)):
        for j in range(i + 1, len(valid_coords)):
            dist = calculate_distance(valid_coords[i], valid_coords[j])
            
            # Check if it's a bond (within covalent radius sum + tolerance)
            atom1, atom2 = valid_atomic_nums[i], valid_atomic_nums[j]
            if atom1 in covalent_radii and atom2 in covalent_radii:
                max_bond_length = (covalent_radii[atom1] + covalent_radii[atom2]) * 1.3
                if dist < max_bond_length:
                    bond_lengths.append(dist)
    
    # Calculate angles for triplets
    if len(valid_coords) >= 3:
        for i in range(len(valid_coords)):
            for j in range(len(valid_coords)):
                if i == j:
                    continue
                for k in range(len(valid_coords)):
                    if k == i or k == j:
                        continue
                    
                    # Check if i-j and j-k are bonds
                    dist_ij = calculate_distance(valid_coords[i], valid_coords[j])
                    dist_jk = calculate_distance(valid_coords[j], valid_coords[k])
                    
                    atom_i, atom_j, atom_k = valid_atomic_nums[i], valid_atomic_nums[j], valid_atomic_nums[k]
                    
                    if (atom_i in covalent_radii and atom_j in covalent_radii and 
                        atom_k in covalent_radii):
                        max_ij = (covalent_radii[atom_i] + covalent_radii[atom_j]) * 1.3
                        max_jk = (covalent_radii[atom_j] + covalent_radii[atom_k]) * 1.3
                        
                        if dist_ij < max_ij and dist_jk < max_jk:
                            angle = calculate_angle(valid_coords[i], valid_coords[j], valid_coords[k])
                            angles.append(angle)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Molecular Geometry Analysis', fontsize=16, fontweight='bold')

# Bond length distribution
if bond_lengths:
    axes[0, 0].hist(bond_lengths, bins=30, alpha=0.7, color='green', edgecolor='black')
    axes[0, 0].set_xlabel('Bond Length (Å)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Bond Length Distribution')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Add statistics
    mean_bond = np.mean(bond_lengths)
    axes[0, 0].axvline(mean_bond, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_bond:.2f} Å')
    axes[0, 0].legend()
else:
    axes[0, 0].text(0.5, 0.5, 'No bonds detected', ha='center', va='center', transform=axes[0, 0].transAxes)
    axes[0, 0].set_title('Bond Length Distribution')

# Angle distribution
if angles:
    axes[0, 1].hist(angles, bins=30, alpha=0.7, color='orange', edgecolor='black')
    axes[0, 1].set_xlabel('Bond Angle (degrees)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Bond Angle Distribution')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Add common angle lines
    for angle, label in [(109.5, 'tetrahedral'), (120, 'trigonal'), (180, 'linear')]:
        axes[0, 1].axvline(angle, color='red', linestyle=':', alpha=0.7, label=label)
    axes[0, 1].legend()
else:
    axes[0, 1].text(0.5, 0.5, 'No angles detected', ha='center', va='center', transform=axes[0, 1].transAxes)
    axes[0, 1].set_title('Bond Angle Distribution')

# Center of mass distance distribution
if center_of_mass_distances:
    axes[1, 0].hist(center_of_mass_distances, bins=30, alpha=0.7, color='purple', edgecolor='black')
    axes[1, 0].set_xlabel('Distance from Center of Mass (Å)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Atomic Distance from Center of Mass')
    axes[1, 0].grid(True, alpha=0.3)

# Geometry statistics summary
stats_text = "Geometry Statistics:\n\n"
if bond_lengths:
    stats_text += f"Bond Lengths:\n"
    stats_text += f"  Mean: {np.mean(bond_lengths):.3f} Å\n"
    stats_text += f"  Std: {np.std(bond_lengths):.3f} Å\n"
    stats_text += f"  Range: [{np.min(bond_lengths):.3f}, {np.max(bond_lengths):.3f}] Å\n\n"

if angles:
    stats_text += f"Bond Angles:\n"
    stats_text += f"  Mean: {np.mean(angles):.1f}°\n"
    stats_text += f"  Std: {np.std(angles):.1f}°\n"
    stats_text += f"  Range: [{np.min(angles):.1f}, {np.max(angles):.1f}]°\n\n"

if center_of_mass_distances:
    stats_text += f"Center of Mass Distances:\n"
    stats_text += f"  Mean: {np.mean(center_of_mass_distances):.3f} Å\n"
    stats_text += f"  Std: {np.std(center_of_mass_distances):.3f} Å\n"

axes[1, 1].text(0.1, 0.9, stats_text, fontsize=10, verticalalignment='top', transform=axes[1, 1].transAxes)
axes[1, 1].axis('off')
axes[1, 1].set_title('Geometry Summary')

plt.tight_layout()
plt.show()

print(f"🔍 Geometry Analysis Results:")
print(f"   Total bonds analyzed: {len(bond_lengths)}")
print(f"   Total angles analyzed: {len(angles)}")
print(f"   Total center distances: {len(center_of_mass_distances)}")

## 🎨 Visualization of Sample Molecules {#visualization}

In [ ]:
# Create visualizer
visualizer = MolecularVisualizer()

# Select diverse sample molecules
sample_molecules = []
size_categories = {'small': [], 'medium': [], 'large': []}

for mol in molecules_data:
    size = mol['num_atoms']
    if size <= 5:
        size_categories['small'].append(mol)
    elif size <= 12:
        size_categories['medium'].append(mol)
    else:
        size_categories['large'].append(mol)

# Select samples from each category
for category, molecules in size_categories.items():
    if molecules:
        sample_molecules.append(np.random.choice(molecules))

print(f"🎨 Visualizing {len(sample_molecules)} sample molecules")

# Create 3D visualizations
for i, mol in enumerate(sample_molecules[:3]):  # Show max 3 molecules
    print(f"\n📊 Molecule {i+1} - {mol['num_atoms']} atoms")
    
    # Create molecule dict for visualizer
    mol_dict = {
        'atomic_nums': mol['atomic_nums'],
        'coordinates': mol['coordinates'],
        'num_atoms': mol['num_atoms']
    }
    
    # Create Plotly visualization
    fig = visualizer.visualize_plotly(mol_dict, show_bonds=True, show_atom_labels=True)
    fig.update_layout(title=f'Sample Molecule {i+1} ({mol["num_atoms"]} atoms)')
    fig.show()
    
    # Print atomic composition
    unique_atoms, counts = np.unique(mol['atomic_nums'], return_counts=True)
    composition = []
    for atom_num, count in zip(unique_atoms, counts):
        element = Chem.GetPeriodicTable().GetElementSymbol(atom_num)
        composition.append(f"{element}{count}")
    
    print(f"   Composition: {''.join(composition)}")
    print(f"   Coordinate range: [{np.min(mol['coordinates']):.3f}, {np.max(mol['coordinates']):.3f}] Å")

## 📊 Statistical Summary {#statistics}

In [ ]:
# Create comprehensive statistical summary
print("📊 COMPREHENSIVE DATASET STATISTICS")
print("=" * 50)

# Dataset overview
print(f"\n📋 Dataset Overview:")
print(f"   Total molecules analyzed: {len(molecules_data):,}")
print(f"   Total atoms: {sum(mol['num_atoms'] for mol in molecules_data):,}")
print(f"   Average atoms per molecule: {np.mean([mol['num_atoms'] for mol in molecules_data]):.2f}")

# Molecular size statistics
sizes = [mol['num_atoms'] for mol in molecules_data]
print(f"\n📏 Molecular Size Statistics:")
print(f"   Mean: {np.mean(sizes):.2f} atoms")
print(f"   Median: {np.median(sizes):.2f} atoms")
print(f"   Standard deviation: {np.std(sizes):.2f} atoms")
print(f"   Min: {np.min(sizes)} atoms")
print(f"   Max: {np.max(sizes)} atoms")
print(f"   25th percentile: {np.percentile(sizes, 25):.2f} atoms")
print(f"   75th percentile: {np.percentile(sizes, 75):.2f} atoms")

# Atomic composition
print(f"\n⚛️ Atomic Composition:")
total_atoms = sum(atom_counts.values())
for atomic_num, count in sorted(atom_counts.items()):
    element = Chem.GetPeriodicTable().GetElementSymbol(atomic_num)
    percentage = (count / total_atoms) * 100
    print(f"   {element} (Z={atomic_num}): {count:,} atoms ({percentage:.1f}%)")

# Coordinate statistics
print(f"\n📐 Coordinate Statistics:")
if len(all_coords) > 0:
    for i, axis in enumerate(['X', 'Y', 'Z']):
        coord_data = all_coords[:, i]
        print(f"   {axis} axis:")
        print(f"     Mean: {np.mean(coord_data):.4f} Å")
        print(f"     Std: {np.std(coord_data):.4f} Å")
        print(f"     Range: [{np.min(coord_data):.4f}, {np.max(coord_data):.4f}] Å")

# Geometry statistics
print(f"\n🔍 Geometry Statistics:")
if bond_lengths:
    print(f"   Bond lengths:")
    print(f"     Mean: {np.mean(bond_lengths):.4f} Å")
    print(f"     Std: {np.std(bond_lengths):.4f} Å")
    print(f"     Range: [{np.min(bond_lengths):.4f}, {np.max(bond_lengths):.4f}] Å")
    print(f"     Total bonds: {len(bond_lengths):,}")

if angles:
    print(f"   Bond angles:")
    print(f"     Mean: {np.mean(angles):.2f}°")
    print(f"     Std: {np.std(angles):.2f}°")
    print(f"     Range: [{np.min(angles):.2f}, {np.max(angles):.2f}]°")
    print(f"     Total angles: {len(angles):,}")

# Recommendations for model design
print(f"\n💡 Model Design Recommendations:")
print(f"   Max atoms per molecule: {np.max(sizes)} (use {np.max(sizes)} as model capacity)")
print(f"   Recommended latent dim: 16-32 (based on molecular complexity)")
print(f"   Coordinate normalization: z-score (mean={np.mean(all_coords):.3f}, std={np.std(all_coords):.3f})")
print(f"   Atom types: {list(atom_counts.keys())} (H, C, N, O, F)")
print(f"   Recommended batch size: 16-32 (depending on GPU memory)")

print(f"\n✅ Exploratory Data Analysis Complete!")

## 🎯 Key Insights and Findings

### Molecular Characteristics
- **Size Distribution**: Most molecules contain 5-15 atoms, with a peak around 8-10 atoms
- **Atomic Composition**: Dominated by Carbon and Hydrogen, with significant Nitrogen and Oxygen presence
- **Geometry**: Bond lengths and angles follow expected chemical patterns

### Data Quality
- **Coordinate Range**: Molecules are reasonably sized, spanning ~5-10 Å in each dimension
- **Structural Diversity**: Good variety of molecular sizes and compositions
- **Normalization**: Coordinates benefit from z-score normalization

### Model Architecture Recommendations
- **Input Size**: Based on max atoms × (atom_types + 3_coordinates)
- **Latent Space**: 16-32 dimensions should capture molecular diversity
- **Processing**: Coordinate normalization and atom type encoding are essential
- **Batch Size**: 16-32 molecules per batch for stable training

This analysis provides a solid foundation for designing and training the molecular VAE model.